In [3]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# 设置matplotlib为黑色主题
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': 'black',
    'axes.facecolor': 'black',
    'axes.edgecolor': 'white',
    'axes.labelcolor': 'white',
    'text.color': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'grid.color': '#444444',
    'grid.alpha': 0.3,
})

# <font color="#00FFFF" >**第四章 卷积神经网络 - 课堂练习**</font>

## <font color="#FF6B00" >**练习1：感受野计算**</font>

#### <font color="#CCFF00" >**练习目标**</font>

通过本练习，你将：
- 掌握感受野递推公式的应用
- 理解网络深度与感受野大小的关系
- 体会 "小卷积核 + 深层网络" 的设计思想

### **<font color="#39FF14" size=6 >任务描述</font>**

给定以下网络结构，请手工计算每一层神经元在输入图像上的感受野大小：

| 层号 | 操作       | 核大小 $k$ | 步长 $s$ |
|:----:|:----------:|:----------:|:--------:|
| 1    | 3x3 卷积   | 3          | 1        |
| 2    | 2x2 池化   | 2          | 2        |
| 3    | 3x3 卷积   | 3          | 1        |
| 4    | 2x2 池化   | 2          | 2        |
| 5    | 3x3 卷积   | 3          | 1        |

#### <font color="#CCFF00" >**递推公式（从后往前回溯）**</font>

$$RF_{l-1} = (RF_l - 1) \times s_l + k_l$$

其中：
- $RF_l$：第 $l$ 层一个神经元在特征图上的感受野边长
- $s_l$：第 $l$ 层的步长
- $k_l$：第 $l$ 层的卷积核或池化窗口大小

#### <font color="#CCFF00" >**要求**</font>

<font color="#FF6B00" >**任务 1.1**</font>：填写下表，计算每层在输入图像上的感受野

| 层号 | 当前层感受野（边长） | 计算公式 |
|:----:|:--------------------:|:---------|
| 5    | $1 \times 1$        |      |
| 4    | $3 \times 3$        |        |
| 3    | $6 \times 6$        |        |
| 2    | $8 \times 8$        |         |
| 1    | $16 \times 16$        |         |
| 输入 | $18 \times 18$        |         |


<font color="#FF6B00" >**任务 1.2**</font>：思考并回答

1. 第 5 层的一个神经元最终"看到"了输入图像多大区域？
2. 这说明了什么？（提示：与卷积核大小和网络深度的关系）

<font color="#FF6B00" >**任务 1.3**</font>：拓展思考

如果有一个网络，只有 1 个卷积层，卷积核大小为 $19 \times 19$，步长为 1。

- 该层神经元的感受野是 $19 \times 19$
- 参数量为 $19 \times 19 = 361$（假设输入通道为 1，输出通道为 1，不考虑偏置）

对比练习中的 5 层网络（其中包含 **3 层 3x3 卷积层**，2 层池化层无参数）：
- 第 5 层感受野达到了 $18 \times 18$（接近 $19 \times 19$）
- **3 层** 3x3 卷积的总参数量为 $3 \times (3 \times 3) = 27$（不考虑偏置）

<font color="#FF00A0" >**问题**</font>：为什么 "多个小卷积核堆叠" 比 "单个大卷积核" 更优？
从**参数量**和**非线性表达能力**两个角度分析。

<font color="#CCFF00" >**提示**</font>：注意网络中只有卷积层有参数，池化层（第 2、4 层）没有可学习的参数。

---

## <font color="#FF6B00" >**练习2：简易CNN架构设计**</font>

#### <font color="#CCFF00" >**练习目标**</font>

通过本练习，你将：
- 学会设计一个简单的CNN架构
- 掌握特征图尺寸变化的计算方法
- 学会估算网络参数量
- 理解通道数变化的设计理念

### **<font color="#39FF14" size=6 >任务描述</font>**

假设你要设计一个简单的CNN来对手写数字（MNIST数据集）进行分类：

<font color="#FF00A0" >**数据集信息**</font>：
- 图像尺寸：$28 \times 28$ 像素
- 通道数：1（灰度图像）
- 类别数：10（数字 0-9）

<font color="#FF00A0" >**设计要求**</font>：
- 至少包含 **2 个卷积块**
- 每个卷积块包含：**卷积层 + 激活函数 + 池化层**
- 最后连接全连接层输出 10 个类别
- 所有卷积使用 **Valid** 形式（无填充）

#### <font color="#CCFF00" >**任务 2.1：完成架构设计表**</font>

请填写下表，设计你的网络结构：

| 层号 | 操作类型 | 输入尺寸       | 输出尺寸       | 核大小/说明        |
|:----:|:--------:|:--------------:|:--------------:|:------------------:|
| 0    | 输入     | -              | $1 \times 28 \times 28$ | 原始图像           |
| 1    | Conv     | $1 \times 28 \times 28$ | 16 x 26 x 26 | $3 \times 3$, 16通道 |
| 2    | ReLU     | 16 x 26 x 26   | 16 x 26 x 26 | 激活函数           |
| 3    | MaxPool  | 16 x 26 x 26   | 16 x 13 x 13   | $2 \times 2$      |
| 4    | Conv     | 16 x 13 x 13   | 32 x 11 x 11   | $3 \times 3$, 32通道 |
| 5    | ReLU     | 32 x 11 x 11   | 32 x 11 x 11 | 激活函数           |
| 6    | MaxPool  | 32 x 11 x 11   | 32 x 5 x 5     | $2 \times 2$      |
| 7    | Flatten  | 32 x 5 x 5     | 800            | 展平               |
| 8    | Linear   | 800          | 128            | 全连接             |
| 9    | ReLU     | 128            | 128            | 激活函数           |
| 10   | Linear   | 128            | 10             | 输出层             |

<font color="#FF00A0" >**计算提示**</font>：

**卷积层输出尺寸公式（Valid）**：
$$
H_{out} = H_{in} - k + 1 \\ 
W_{out} = W_{in} - k + 1
$$

**池化层输出尺寸公式**：
$$
H_{out} = \left\lfloor \frac{H_{in} - k}{s} \right\rfloor + 1 \\ 
W_{out} = \left\lfloor \frac{W_{in} - k}{s} \right\rfloor + 1
$$
其中 $k$ 是池化核大小，$s$ 是步长（通常 $s = k$）

#### <font color="#CCFF00" >**任务 2.2：估算参数量**</font>

<font color="#FF6B00" >**卷积层参数量计算公式**</font>：
$$
\text{Params}_{conv} = (C_{in} \times k \times k) \times C_{out} + C_{out}
$$
其中 $C_{in}$ 是输入通道数，$C_{out}$ 是输出通道数，$k$ 是卷积核大小，$+ C_{out}$ 是偏置项。

<font color="#FF6B00" >**全连接层参数量计算公式**</font>：
$$
\text{Params}_{fc} = (N_{in} \times N_{out}) + N_{out}
$$
其中 $N_{in}$ 是输入神经元数，$N_{out}$ 是输出神经元数，$+ N_{out}$ 是偏置项。

请计算：
1. 第 1 层卷积的参数量：__1x3x3x16+16=160__
2. 第 4 层卷积的参数量：___16x3x3x32+32=4640___
3. 第 8 层全连接的参数量：____800x128=102400____
4. 第 10 层全连接的参数量：___128+10=138___
5. **总参数量**：___160+4640+102400+138 = 108618___


#### <font color="#CCFF00" >**任务 2.3：设计思考**</font>

请回答以下问题：

1. **为什么第一层用 16 个通道，第二层用 32 个通道？**
   （提示：考虑特征复杂度和层次抽象）

2. **为什么池化层能减小特征图尺寸但增大感受野？**

3. **如果将两个卷积层的通道数互换（第一层 32，第二层 16），会有什么影响？**

#### <font color="#CCFF00" >**任务 2.4：挑战题（选做）**</font>

<font color="#FF00A0" >**设计一个更精简的网络**</font>：

要求：
- 最终感受野至少要覆盖输入图像的 $20 \times 20$ 区域（用于捕捉整体形状）
- 总参数量控制在 **50,000** 以内
- 仍然保持 2 个卷积块的基本结构

你可以调整：卷积核大小、通道数、是否使用更多池化层等。

请画出你的网络结构，并说明设计思路。

In [5]:
# 任务 2.5：用 PyTorch 实现你设计的网络（选做）

import torch.nn as nn

class MyCNN(nn.Module):
    def __init__(self):
        super(MyCNN, self).__init__()
        # 提示1：使用 nn.Sequential 组织特征提取器（卷积块）
        # 提示2：每个卷积块包含：Conv2d → ReLU → MaxPool2d
        # 提示3：记得计算 Flatten 后的维度（通道数 × 高 × 宽）
        # 提示4：分类器使用 nn.Linear 和 ReLU
        
        # 在这里定义你的网络层
        # self.features = nn.Sequential(...)
        # self.classifier = nn.Sequential(...)
        pass
    
    def forward(self, x):
        # 在这里定义前向传播
        # x = self.features(x)
        # x = self.classifier(x)
        # return x
        pass

# 创建模型实例
# model = MyCNN()

# 统计参数量
# total_params = sum(p.numel() for p in model.parameters())
# print(f"总参数量: {total_params}")

---

## <font color="#FFEA00" >**练习完成！**</font>
